In [77]:
# Import necessary libraries for data manipulation and modeling
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [78]:
# Load the F1 Strategy Dataset
df = pd.read_csv('f1_strategy_dataset_v4.csv')

# Display the first few rows
df.head()

,Driver,LapNumber,Compound,Stint,TyreLife,Position,LapTime (s),Race,Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
0,ALB,1,MEDIUM,1,2.0,17,100.625,Abu Dhabi Grand Prix,2023,0.000,0.000,0,0,0.017241,0.117647,0.0
1,ALB,2,MEDIUM,1,3.0,18,93.560,Abu Dhabi Grand Prix,2023,-7.065,-7.065,0,0,0.034483,0.176471,-1.0
2,ALB,3,MEDIUM,1,4.0,18,91.768,Abu Dhabi Grand Prix,2023,-1.792,-8.857,0,0,0.051724,0.235294,0.0
3,ALB,4,MEDIUM,1,5.0,18,91.591,Abu Dhabi Grand Prix,2023,-0.177,-9.034,0,0,0.068966,0.294118,0.0
4,ALB,5,MEDIUM,1,6.0,18,91.422,Abu Dhabi Grand Prix,2023,-0.169,-9.203,0,0,0.086207,0.352941,0.0


In [79]:
# Basic statistics
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 101371 entries, 0 to 101370
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Driver                  101371 non-null  str    
 1   LapNumber               101371 non-null  int64  
 2   Compound                101305 non-null  str    
 3   Stint                   101371 non-null  int64  
 4   TyreLife                101371 non-null  float64
 5   Position                101371 non-null  int64  
 6   LapTime (s)             101371 non-null  float64
 7   Race                    101371 non-null  str    
 8   Year                    101371 non-null  int64  
 9   LapTime_Delta           101371 non-null  float64
 10  Cumulative_Degradation  101371 non-null  float64
 11  PitStop                 101371 non-null  int64  
 12  PitNextLap              101371 non-null  int64  
 13  RaceProgress            101371 non-null  float64
 14  Normalized_TyreLife     101371 

In [80]:
df.describe()

,LapNumber,Stint,TyreLife,Position,LapTime (s),Year,LapTime_Delta,Cumulative_Degradation,PitStop,PitNextLap,RaceProgress,Normalized_TyreLife,Position_Change
count,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000,101371.000000
mean,30.444841,2.046394,14.549339,9.759132,92.587188,2023.589685,-0.203891,-29.550051,0.251581,0.254797,0.432618,0.386521,-0.004636
std,18.146942,0.948797,10.313385,5.406456,33.231414,1.098518,45.344910,70.235759,0.433924,0.435749,0.258129,0.259906,3.912725
min,1.000000,1.000000,1.000000,1.000000,67.012000,2022.000000,-2403.895000,-274.564000,0.000000,0.000000,0.012821,0.012821,-18.000000
25%,15.000000,1.000000,7.000000,5.000000,82.021000,2023.000000,-7.253000,-51.054500,0.000000,0.000000,0.210526,0.172414,-2.000000
50%,30.000000,2.000000,13.000000,10.000000,91.167000,2024.000000,-0.027000,-21.678000,0.000000,0.000000,0.421053,0.333333,0.000000
75%,45.000000,3.000000,20.000000,14.000000,99.356000,2025.000000,5.705000,-3.725500,1.000000,1.000000,0.631579,0.562500,2.000000
max,78.000000,8.000000,78.000000,20.000000,2526.253000,2025.000000,2433.472000,2412.431000,1.000000,1.000000,1.000000,1.000000,18.000000


In [81]:

# Target distribution
df['PitNextLap'].value_counts()
df['PitNextLap'].value_counts(normalize=True)



PitNextLap
0    0.745203
1    0.254797
Name: proportion, dtype: float64

In [82]:
# Missing values
df.isnull().sum()

Driver                     0
LapNumber                  0
Compound                  66
Stint                      0
TyreLife                   0
Position                   0
LapTime (s)                0
Race                       0
Year                       0
LapTime_Delta              0
Cumulative_Degradation     0
PitStop                    0
PitNextLap                 0
RaceProgress               0
Normalized_TyreLife        0
Position_Change            0
dtype: int64

In [83]:
# Time-series aware train/test split

# Separate data by year
train_df = df[df['Year'].isin([2022, 2023])].copy()
val_df = df[df['Year'] == 2024].copy()
test_df = df[df['Year'] == 2025].copy()

print(f"Total rows: {len(df)}")
print(f"Train size: {len(train_df)} ({len(train_df)/len(df):.2%})")
print(f"Val size: {len(val_df)} ({len(val_df)/len(df):.2%})")
print(f"Test size: {len(test_df)} ({len(test_df)/len(df):.2%})")
print(f"\nTrain/Val/Test ratio: {len(train_df)/len(val_df):.2f} / {len(train_df)/len(test_df):.2f}")

# Verify temporal order is preserved
print(f"\nUnique sequences in train: {train_df.groupby(['Driver', 'Race', 'Year']).ngroups}")
print(f"Unique sequences in val: {val_df.groupby(['Driver', 'Race', 'Year']).ngroups}")
print(f"Unique sequences in test: {test_df.groupby(['Driver', 'Race', 'Year']).ngroups}")

Total rows: 101371
Train size: 46774 (46.14%)
Val size: 27557 (27.18%)
Test size: 27040 (26.67%)

Train/Val/Test ratio: 1.70 / 1.73

Unique sequences in train: 878
Unique sequences in val: 485
Unique sequences in test: 485


In [84]:
# Create lagged features for Position, LapTime, and degradation metrics
# Group by Driver, Race, Year to maintain temporal order within sequences

# Create lagged features
train_df['Position_lag1'] = train_df.groupby(['Driver', 'Race', 'Year'])['Position'].shift(1)
train_df['Position_lag2'] = train_df.groupby(['Driver', 'Race', 'Year'])['Position'].shift(2)
train_df['Position_lag3'] = train_df.groupby(['Driver', 'Race', 'Year'])['Position'].shift(3)

# LapTime lags
train_df['LapTime_lag1'] = train_df.groupby(['Driver', 'Race', 'Year'])['LapTime (s)'].shift(1)
train_df['LapTime_lag2'] = train_df.groupby(['Driver', 'Race', 'Year'])['LapTime (s)'].shift(2)
train_df['LapTime_lag3'] = train_df.groupby(['Driver', 'Race', 'Year'])['LapTime (s)'].shift(3)

# Rolling average pace (using LapTime_Delta as a proxy for pace change)
# Rolling average of LapTime_Delta over 3 laps
train_df['Rolling_avg_pace'] = train_df.groupby(['Driver', 'Race', 'Year'])['LapTime_Delta'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Rolling degradation (rolling average of Cumulative_Degradation)
train_df['Rolling_degradation'] = train_df.groupby(['Driver', 'Race', 'Year'])['Cumulative_Degradation'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

# Fill NaN values (from shifts) with 0
# Only fill numeric columns with 0, leave string/categorical columns unchanged
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
train_df[numeric_cols] = train_df[numeric_cols].fillna(0)


In [85]:
train_df = train_df.drop(['Driver', 'Race'], axis=1)

In [86]:
# Convert Compound column to categorical with specified order
categories = ['SOFT', 'MEDIUM', 'HARD']
dtype = pd.CategoricalDtype(categories=categories, ordered=True)
train_df['Compound'] = train_df['Compound'].astype(dtype)

# Verify the conversion
train_df['Compound'].cat.categories.tolist()
train_df['Compound'].cat.codes.value_counts()

 2    19612
 1    17357
 0     7349
-1     2456
Name: count, dtype: int64

In [87]:
# Impute missing values
imputer = SimpleImputer(strategy='most_frequent')
train_df['Compound'] = imputer.fit_transform(train_df['Compound'].values.reshape(-1, 1)).flatten()

In [88]:
# Encode Compound column with OrdinalEncoder
encoder = OrdinalEncoder()
train_df['Compound_encoded'] = encoder.fit_transform(train_df['Compound'].to_numpy().reshape(-1, 1)).flatten()

# Drop the original Compound column
train_df.drop('Compound', axis=1, inplace=True)

In [89]:
X, y = train_df.drop('PitNextLap', axis=1), train_df['PitNextLap']

In [90]:
X

,LapNumber,Stint,TyreLife,Position,LapTime (s),Year,LapTime_Delta,Cumulative_Degradation,PitStop,RaceProgress,...,Position_Change,Position_lag1,Position_lag2,Position_lag3,LapTime_lag1,LapTime_lag2,LapTime_lag3,Rolling_avg_pace,Rolling_degradation,Compound_encoded
0,1,1,2.0,17,100.625,2023,0.000,0.000,0,0.017241,...,0.0,0.0,0.0,0.0,0.000,0.000,0.000,0.000000,0.000000,1.0
1,2,1,3.0,18,93.560,2023,-7.065,-7.065,0,0.034483,...,-1.0,17.0,0.0,0.0,100.625,0.000,0.000,-3.532500,-3.532500,1.0
2,3,1,4.0,18,91.768,2023,-1.792,-8.857,0,0.051724,...,0.0,18.0,17.0,0.0,93.560,100.625,0.000,-2.952333,-5.307333,1.0
3,4,1,5.0,18,91.591,2023,-0.177,-9.034,0,0.068966,...,0.0,18.0,18.0,17.0,91.768,93.560,100.625,-3.011333,-8.318667,1.0
4,5,1,6.0,18,91.422,2023,-0.169,-9.203,0,0.086207,...,0.0,18.0,18.0,18.0,91.591,91.768,93.560,-0.712667,-9.031333,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101366,52,3,29.0,12,103.373,2022,26.850,-90.264,0,0.722222,...,2.0,12.0,11.0,11.0,104.328,102.831,103.603,25.680000,-90.126333,0.0
101367,53,3,30.0,13,105.200,2022,27.061,-88.437,0,0.736111,...,1.0,12.0,12.0,11.0,103.373,104.328,102.831,26.409333,-89.336667,0.0
101368,54,3,31.0,13,104.102,2022,27.149,-89.535,0,0.750000,...,0.0,13.0,12.0,12.0,105.200,103.373,104.328,27.020000,-89.412000,0.0
101369,55,3,32.0,14,103.812,2022,4.428,-89.825,0,0.763889,...,-1.0,13.0,13.0,12.0,104.102,105.200,103.373,19.546000,-89.265667,0.0
